# Noisy 1D differentiation: local Chebyshev (LDC) endpoints

Compare the default finite-difference endpoints with the existing local
Chebyshev endpoint estimator on identical noisy samples. Here **LDC** refers
to the archive's *low-degree Chebyshev* endpoint fit, not Full-LDC or an iterative
local-defect-correction algorithm.

The nonperiodic signal is $f(x)=e^{0.2x}+\sin(2.3x)+0.2\cos(5.1x)$ on $[0,2\pi]$.
Noise is independent Gaussian noise with standard deviation $\sigma\,\mathrm{RMS}(f)$.
Use 64 fixed realizations shared by every method and noise level. Both endpoints
are noisy. We hold spline degree, constraints, knots, regularization and Fourier
operation fixed; only the endpoint estimator changes. There is **no Fourier
filter or whole-signal denoising**.

Install `python -m pip install -e './jax[notebook]'` from the repository root.
On the development Mac, launch with the corrected BLAS as described in
`docs/corrected_blas_build.md`, or use `OMP_NUM_THREADS=1` with the original build.


In [ ]:
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import bspf_jax as bspf

x = jnp.linspace(0, 2*jnp.pi, 513)
f = jnp.exp(.2*x) + jnp.sin(2.3*x) + .2*jnp.cos(5.1*x)
exact = .2*jnp.exp(.2*x) + 2.3*jnp.cos(2.3*x) - 1.02*jnp.sin(5.1*x)
sigmas = np.array([0., 1e-10, 1e-8, 1e-6, 1e-4, 1e-2])
noise = jnp.asarray(np.random.default_rng(20260917).standard_normal((x.size, 64)))
scale = jnp.sqrt(jnp.mean(f**2))
options = dict(degree=9, n_basis=18, constraint_order=8, lam=1e-6)
plans = {
    "FD9": bspf.plan_1d(x, **options, boundary_points=9),
    "LDC M12/P16": bspf.plan_1d(x, **options, endpoint_method="chebyshev",
                                chebyshev_modes=12, boundary_points=16),
    "LDC M8/P40": bspf.plan_1d(x, **options, endpoint_method="chebyshev",
                               chebyshev_modes=8, boundary_points=40),
}
derivative = jax.jit(bspf.differentiate)
boundary = np.r_[np.arange(40), np.arange(x.size-40, x.size)]
interior = np.arange(40, x.size-40)


## Paired noise study

Total error is the mean over realizations of $\|D f_\sigma-f'\|_2/\|f'\|_2$.
Noise gain is $\|D f_\sigma-D f\|_2/\|f_\sigma-f\|_2$ and has units of inverse
length. Boundary/interior gains use the same 40-point boundary mask for every
method. The broad M8/P40 fit tests a noise-versus-bias tradeoff; it is not a new
default. All Chebyshev fits retain the default modal penalty alpha $10^{-12}$.


In [ ]:
results = {}
for name, plan in plans.items():
    clean = derivative(plan, f)
    totals, gains, profiles = [], [], []
    for sigma in sigmas:
        perturbation = sigma*scale*noise
        noisy = derivative(plan, f[:, None] + perturbation)
        response = noisy-clean[:, None]
        totals.append(np.asarray(jnp.linalg.norm(noisy-exact[:, None], axis=0)
                                 / jnp.linalg.norm(exact)))
        profiles.append(np.asarray(jnp.sqrt(jnp.mean(response**2, axis=1))))
        if sigma:
            gains.append([float(jnp.mean(jnp.linalg.norm(response[ids], axis=0)
                                        / jnp.linalg.norm(perturbation[ids], axis=0)))
                          for ids in (np.arange(x.size), boundary, interior)])
        assert bool(jnp.all(jnp.isfinite(noisy)))
    results[name] = dict(clean=float(jnp.linalg.norm(clean-exact)/jnp.linalg.norm(exact)),
                         total=np.array(totals), gain=np.array(gains), rms=np.array(profiles))
    assert results[name]["clean"] < 1e-5
    print(f"{name}: clean relative L2={results[name]['clean']:.3e}; "
          f"noise gain (all/boundary/interior)={results[name]['gain'][3]}")
print("sigma", *plans)
for i, sigma in enumerate(sigmas):
    print(f"{sigma:.0e}", *(f"{results[name]['total'][i].mean():.3e}" for name in plans))


## Total error and spatial noise response

Shading shows the 10th–90th percentiles across realizations, not confidence
intervals. Horizontal dashed lines mark clean-signal error. The right panel
shows RMS derivative noise response at relative input noise $10^{-4}$. Endpoint
estimation can reduce boundary artifacts, but differentiating the unfiltered
Fourier residual still amplifies noise throughout the domain.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
for name, result in results.items():
    line, = axes[0].loglog(sigmas[1:], result["total"][1:].mean(axis=1), "o-", label=name)
    low, high = np.percentile(result["total"][1:], [10, 90], axis=1)
    axes[0].fill_between(sigmas[1:], low, high, color=line.get_color(), alpha=.15)
    axes[0].axhline(result["clean"], color=line.get_color(), ls="--", alpha=.5)
    axes[1].semilogy(x, result["rms"][4], label=name)
axes[0].set(xlabel="Input noise standard deviation / RMS(f)", ylabel="Derivative relative L2 error")
axes[1].set(xlabel="x", ylabel="RMS derivative noise response (sigma=1e-4)")
for ax in axes:
    ax.legend(); ax.grid(True, which="both", alpha=.2)
plt.show()
